In [1]:
import os
import time
import sqlalchemy as sa
import pandas as pd
import numpy as np

# Ensure working directory is the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print(f"Current Working Directory: {os.getcwd()}")

# Create artifacts directory structure if not exists
os.makedirs('artifacts/data', exist_ok=True)
os.makedirs('artifacts/figures', exist_ok=True)
os.makedirs('artifacts/models', exist_ok=True)
os.makedirs('artifacts/reports', exist_ok=True)


Current Working Directory: c:\Users\rosto\OneDrive\سطح المكتب\مشروع mlops


## 1. PostgreSQL (Database Connection)

In [2]:
# Load credentials securely from .env file
import os
from dotenv import load_dotenv

load_dotenv()

DB_USER = os.getenv('DB_USER', 'postgres')
DB_PASSWORD = os.getenv('DB_PASSWORD', '')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'olist')

DB_URL = f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = sa.create_engine(DB_URL)

# Verify connection and list tables
with engine.connect() as conn:
    query = sa.text("SELECT table_name FROM information_schema.tables WHERE table_schema='public' ORDER BY table_name;")
    tables = [row[0] for row in conn.execute(query).fetchall()]

print(f" Successfully connected to PostgreSQL! Found {len(tables)} tables:")
for t in tables:
    print(f"  - {t}")


 Successfully connected to PostgreSQL! Found 9 tables:
  - customers
  - geolocation
  - order_items
  - order_payments
  - order_reviews
  - orders
  - product_category_translation
  - products
  - sellers


## 2. Individual Tables Inspection

In [3]:
# Load all tables into memory and display basic diagnostics
table_dfs = {}
t0 = time.time()

with engine.connect() as conn:
    for t in tables:
        df = pd.read_sql(f"SELECT * FROM {t}", conn)
        table_dfs[t] = df
        print(f" Table '{t}': {df.shape[0]:,} rows, {df.shape[1]} columns")

print(f"\n  All tables loaded in {time.time() - t0:.2f} seconds.")


 Table 'customers': 99,441 rows, 5 columns
 Table 'geolocation': 1,000,163 rows, 5 columns
 Table 'order_items': 112,650 rows, 7 columns
 Table 'order_payments': 103,886 rows, 5 columns
 Table 'order_reviews': 99,224 rows, 7 columns
 Table 'orders': 99,441 rows, 8 columns
 Table 'product_category_translation': 71 rows, 2 columns
 Table 'products': 32,951 rows, 9 columns
 Table 'sellers': 3,095 rows, 4 columns

  All tables loaded in 10.79 seconds.


In [4]:
# Display Granularity and Primary Key Checks for each core table
granularity_info = {
'orders':     ('order_id', 'Represents a single independent order placed by a customer'),
'customers':  ('customer_id', 'Represents customer account information tied to a specific order (unique per order)'),
'order_items':('order_id + order_item_id', 'Represents a single item within an order (an order may contain multiple items)'),
'order_payments': ('order_id + payment_sequential', 'Represents a payment transaction/installment for an order (an order may have multiple payment records)'),
'order_reviews':  ('review_id + order_id', 'Represents a customer satisfaction review and score submitted post-delivery'),
'products': ('product_id', 'Represents a catalog product with its technical specifications, weight, and dimensions'),
'sellers':  ('seller_id', 'Represents a registered seller/merchant along with their geographic location'),
'geolocation': ('geolocation_zip_code_prefix', 'Represents geographic coordinates (latitude and longitude) for postal zip code prefixes'),
    'product_category_translation': ('product_category_name', 'Lookup table translating product category names from Portuguese to English')}

summary_rows = []
for t, (key, desc) in granularity_info.items():
    df = table_dfs[t]
    dups = df.duplicated().sum()
    summary_rows.append({'Table': t,'Rows': f"{len(df):,}",'Columns': df.shape[1],'Primary Key / Identifier': key,'Duplicate Rows': dups,'Row Granularity (معنى الصف)': desc})

pd.DataFrame(summary_rows)


,Table,Rows,Columns,Primary Key / Identifier,Duplicate Rows,Row Granularity (معنى الصف)
0,orders,"99,441",8,order_id,0,Represents a single independent order placed b...
1,customers,"99,441",5,customer_id,0,Represents customer account information tied t...
2,order_items,"112,650",7,order_id + order_item_id,0,Represents a single item within an order (an o...
3,order_payments,"103,886",5,order_id + payment_sequential,0,Represents a payment transaction/installment f...
4,order_reviews,"99,224",7,review_id + order_id,0,Represents a customer satisfaction review and ...
5,products,"32,951",9,product_id,0,Represents a catalog product with its technica...
6,sellers,"3,095",4,seller_id,0,Represents a registered seller/merchant along ...
7,geolocation,"1,000,163",5,geolocation_zip_code_prefix,261831,Represents geographic coordinates (latitude an...
8,product_category_translation,71,2,product_category_name,0,Lookup table translating product category name...


## 3. Aggregation Before Joining

In [5]:
# 1. Translate product categories and compute product volume
df_products = table_dfs['products'].copy()
df_trans = table_dfs['product_category_translation'].copy()
df_products = df_products.merge(df_trans, on='product_category_name', how='left')
df_products['product_category_name_english'] = df_products['product_category_name_english'].fillna(df_products['product_category_name']).fillna('other')
df_products['volume_cm3'] = df_products['product_length_cm'] * df_products['product_height_cm'] * df_products['product_width_cm']

# 2. Enrich order_items with product and seller details
df_items = table_dfs['order_items'].copy()
df_sellers = table_dfs['sellers'].copy()

items_enriched = df_items.merge(df_products, on='product_id', how='left')
items_enriched = items_enriched.merge(df_sellers, on='seller_id', how='left')

# Sort by order_id and price descending to easily capture primary product/seller
items_enriched = items_enriched.sort_values(by=['order_id', 'price'], ascending=[True, False])

# Aggregate items to 1 row per order_id
items_agg = items_enriched.groupby('order_id').agg(
    order_items_count=('order_item_id', 'count'),
    total_price=('price', 'sum'),
    avg_price=('price', 'mean'),
    max_price=('price', 'max'),
    total_freight=('freight_value', 'sum'),
    avg_freight=('freight_value', 'mean'),
    max_freight=('freight_value', 'max'),
    num_unique_products=('product_id', 'nunique'),
    num_unique_sellers=('seller_id', 'nunique'),
    total_weight_g=('product_weight_g', 'sum'),
    max_weight_g=('product_weight_g', 'max'),
    total_volume_cm3=('volume_cm3', 'sum'),
    primary_product_category=('product_category_name_english', 'first'),
    seller_id=('seller_id', 'first'),
    seller_zip_code_prefix=('seller_zip_code_prefix', 'first'),
    seller_city=('seller_city', 'first'),
    seller_state=('seller_state', 'first')).reset_index()

print(f" Items aggregated successfully: {items_agg.shape[0]:,} orders aggregated from {len(df_items):,} items.")


 Items aggregated successfully: 98,666 orders aggregated from 112,650 items.


In [6]:
# 3. Aggregate Payments
df_payments = table_dfs['order_payments'].copy()
payments_sorted = df_payments.sort_values(by=['order_id', 'payment_value'], ascending=[True, False])

payments_agg = payments_sorted.groupby('order_id').agg(
    total_payment_value=('payment_value', 'sum'),
    max_payment_installments=('payment_installments', 'max'),
    payment_records_count=('payment_sequential', 'count'),
    dominant_payment_type=('payment_type', 'first')).reset_index()

print(f" Payments aggregated successfully: {payments_agg.shape[0]:,} orders from {len(df_payments):,} payment rows.")

# 4. Aggregate Reviews
df_reviews = table_dfs['order_reviews'].copy()
reviews_agg = df_reviews.groupby('order_id').agg(
    review_score=('review_score', 'mean'),
    review_count=('review_id', 'count')).reset_index()

# 5. Geolocation Mean Lookup
df_geo = table_dfs['geolocation'].copy()
geo_lookup = df_geo.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean().reset_index()
print(f" Geolocation aggregated: {geo_lookup.shape[0]:,} unique zip code prefixes.")


 Payments aggregated successfully: 99,440 orders from 103,886 payment rows.
 Geolocation aggregated: 19,015 unique zip code prefixes.


## 4. Executing Final Order-Level Join


In [7]:
df_orders = table_dfs['orders'].copy()
df_customers = table_dfs['customers'].copy()

# Join customers
ml_table = df_orders.merge(df_customers, on='customer_id', how='left')

# Join aggregated items, payments, and reviews
ml_table = ml_table.merge(items_agg, on='order_id', how='left')
ml_table = ml_table.merge(payments_agg, on='order_id', how='left')
ml_table = ml_table.merge(reviews_agg, on='order_id', how='left')

# Attach coordinates for customer and seller
ml_table = ml_table.merge(
    geo_lookup.rename(columns={
        'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
        'geolocation_lat': 'customer_lat',
        'geolocation_lng': 'customer_lng'}),
    on='customer_zip_code_prefix',
    how='left')

ml_table = ml_table.merge(
    geo_lookup.rename(columns={
        'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
        'geolocation_lat': 'seller_lat',
        'geolocation_lng': 'seller_lng'}),
    on='seller_zip_code_prefix',
    how='left')

# Compute Haversine distance in kilometers
def haversine_distance(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6367.0 * c
    return km

ml_table['distance_km'] = haversine_distance(
    ml_table['customer_lng'], ml_table['customer_lat'],
    ml_table['seller_lng'], ml_table['seller_lat'])

# Boolean flag: whether customer and seller are in the same state
ml_table['is_same_state'] = (ml_table['customer_state'] == ml_table['seller_state']).astype('Int64')

print(f" Final Joined Table Shape: {ml_table.shape[0]:,} rows, {ml_table.shape[1]} columns")


 Final Joined Table Shape: 99,441 rows, 41 columns


## 5. Sanity & Quality Checks

In [8]:
# 1. Assert row count and uniqueness
assert len(ml_table) == len(df_orders), " Row count mismatch!"
assert ml_table['order_id'].nunique() == len(ml_table), " Duplicate order_ids detected!"

print(" PASSED: Exactly one row per order_id.")
print(f"Total Unique Orders: {ml_table['order_id'].nunique():,}")

# 2. Preview sample data
display_cols = [
    'order_id', 'order_status', 'customer_state', 'seller_state',
    'is_same_state', 'order_items_count', 'total_price', 'total_freight',
    'distance_km', 'dominant_payment_type']
ml_table[display_cols].head(5)


 PASSED: Exactly one row per order_id.
Total Unique Orders: 99,441


,order_id,order_status,customer_state,seller_state,is_same_state,order_items_count,total_price,total_freight,distance_km,dominant_payment_type
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,SP,SP,1,1.0,29.99,8.72,18.564447,voucher
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,BA,SP,0,1.0,118.70,22.76,850.960462,boleto
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,GO,SP,0,1.0,159.90,19.22,514.087696,credit_card
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,RN,MG,0,1.0,45.00,27.20,1821.082261,credit_card
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,SP,SP,1,1.0,19.90,8.72,29.657992,credit_card


In [9]:
# 3. Missing values summary
missing_summary = ml_table.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
missing_df = pd.DataFrame({
    'Missing Count': missing_summary,
    'Percentage (%)': (missing_summary / len(ml_table) * 100).round(2)})

print("Columns with missing values:")
missing_df


Columns with missing values:


,Missing Count,Percentage (%)
order_delivered_customer_date,2965,2.98
order_delivered_carrier_date,1783,1.79
distance_km,1264,1.27
seller_lat,991,1.00
seller_lng,991,1.00
max_weight_g,791,0.80
avg_price,775,0.78
num_unique_products,775,0.78
total_freight,775,0.78
max_freight,775,0.78


## 6. Save Artifact

In [10]:
parquet_path = 'artifacts/data/01_joined_orders.parquet'
csv_path = 'artifacts/data/01_joined_orders.csv'

ml_table.to_parquet(parquet_path, index=False)
ml_table.to_csv(csv_path, index=False)

print(" Artifact saved successfully:")


 Artifact saved successfully:
